# Rebar DAE Take-Home Case Study

## Setup: Connect to Database

In [ ]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from rapidfuzz import fuzz, process
import re

DB_PATH = 'dae-case/candidate_database.sqlite'
conn = sqlite3.connect(DB_PATH)

def q(sql):
    """Run a SQL query and return a DataFrame."""
    return pd.read_sql(sql, conn)

print(f'Connected to: {DB_PATH}')


## Sanity Checks

In [ ]:
# Row counts across all tables
sanity = q('''
SELECT
    (SELECT COUNT(*)                    FROM dim_projects)               AS projects,
    (SELECT COUNT(DISTINCT company_id)  FROM dim_projects)               AS companies,
    (SELECT COUNT(DISTINCT username)    FROM dim_projects)               AS users,
    (SELECT COUNT(*)                    FROM dim_manufacturers)          AS manufacturers,
    (SELECT COUNT(*)                    FROM dim_equipment_categories)   AS equipment_categories,
    (SELECT COUNT(*)                    FROM fact_project_manufacturers) AS fact_rows,
    (SELECT COUNT(*)                    FROM project_engineers)          AS engineer_rows,
    (SELECT COUNT(*)                    FROM project_architects)         AS architect_rows
''')
print("Table row counts:")
print(sanity.T.rename(columns={0: 'count'}).to_string())


In [ ]:
# Distinct project counts per company
print("Projects per company:")
print(q('''
    SELECT company_id, COUNT(*) AS projects, COUNT(DISTINCT username) AS users
    FROM dim_projects
    GROUP BY company_id
    ORDER BY projects DESC
''').to_string(index=False))


In [ ]:
# Raw vs normalized rows in fact table
print("Fact table: raw vs normalized rows:")
print(q('''
    SELECT
        is_normalized,
        source_type,
        COUNT(*)           AS rows,
        SUM(unit_count)    AS total_units,
        COUNT(DISTINCT project_id) AS projects
    FROM fact_project_manufacturers
    GROUP BY is_normalized, source_type
    ORDER BY is_normalized, source_type
''').to_string(index=False))


In [ ]:
# Date range of the data
print("Data date range:")
print(q('''
    SELECT
        substr(MIN(created_at), 1, 10) AS earliest_project,
        substr(MAX(created_at), 1, 10) AS latest_project
    FROM dim_projects
''').to_string(index=False))

# Market category breakdown
print("\nProjects by market category:")
print(q('''
    SELECT market_category, COUNT(*) AS projects
    FROM dim_projects
    WHERE market_category != ''
    GROUP BY market_category
    ORDER BY projects DESC
''').to_string(index=False))


---
## Part 1: Entity Resolution (Manufacturers)

### 1a) Distinct string counts: raw vs normalized

In [ ]:
counts = q('''
SELECT
    is_normalized,
    COUNT(DISTINCT manufacturer_id)   AS distinct_manufacturer_ids,
    COUNT(DISTINCT m.manufacturer_name) AS distinct_name_strings
FROM fact_project_manufacturers f
JOIN dim_manufacturers m USING (manufacturer_id)
GROUP BY is_normalized
''')
counts['label'] = counts['is_normalized'].map(
    {0: 'Raw (is_normalized=0)', 1: 'Normalized (is_normalized=1)'}
)
print(counts[['label','distinct_manufacturer_ids','distinct_name_strings']].to_string(index=False))


### 1b) Reduction rate: how many raw names were changed?

In [ ]:
pairs = q('''
SELECT
    mr.manufacturer_name AS raw_name,
    mn.manufacturer_name AS norm_name
FROM fact_project_manufacturers r
JOIN fact_project_manufacturers n
    ON  r.project_id   = n.project_id
    AND r.category_id  = n.category_id
    AND r.source_type  = n.source_type
    AND r.is_normalized = 0
    AND n.is_normalized = 1
JOIN dim_manufacturers mr ON r.manufacturer_id = mr.manufacturer_id
JOIN dim_manufacturers mn ON n.manufacturer_id = mn.manufacturer_id
''')

total_raw   = pairs['raw_name'].nunique()
changed     = pairs[pairs['raw_name'] != pairs['norm_name']]
n_changed   = changed['raw_name'].nunique()
n_unchanged = total_raw - n_changed

print(f'Unique raw names with a normalized counterpart : {total_raw:,}')
print(f'  changed by normalization                     : {n_changed:,}  ({n_changed/total_raw*100:.1f}%)')
print(f'  left unchanged                               : {n_unchanged:,}  ({n_unchanged/total_raw*100:.1f}%)')


### 1c) What normalization fixes well: top changed pairs

In [ ]:
fixed = (
    changed.groupby(['raw_name','norm_name'])
           .size()
           .reset_index(name='occurrences')
           .sort_values('occurrences', ascending=False)
)
print('Top 30 normalization changes (by occurrence):')
print(fixed.head(30).to_string(index=False))


### 1c-ii) CRITICAL: Lossy normalization: real names mapped to Undefined and vice versa

In [ ]:
# Cases where a real manufacturer name was mapped TO "Undefined"
real_to_undef = pairs[
    (pairs['norm_name'].str.lower() == 'undefined') &
    (~pairs['raw_name'].str.lower().isin(['undefined','none','','n/a','unknown']))
]
print(f'Real names incorrectly mapped TO "Undefined": {real_to_undef["raw_name"].nunique():,} distinct names')
print(real_to_undef['raw_name'].value_counts().head(20).to_string())

print()

# Cases where "undefined" raw was mapped TO a real manufacturer name
undef_to_real = pairs[
    (pairs['raw_name'].str.lower() == 'undefined') &
    (~pairs['norm_name'].str.lower().isin(['undefined','none','','n/a','unknown']))
]
print(f'\n"undefined" raw incorrectly mapped TO a real name: {undef_to_real["norm_name"].nunique():,} distinct names')
print(undef_to_real['norm_name'].value_counts().head(20).to_string())

# Quantify units affected by lossy mapping
units_real_to_undef = q('''
    SELECT SUM(f.unit_count) AS units
    FROM fact_project_manufacturers f
    JOIN fact_project_manufacturers n
        ON  f.project_id  = n.project_id
        AND f.category_id = n.category_id
        AND f.source_type = n.source_type
        AND f.is_normalized = 0 AND n.is_normalized = 1
    JOIN dim_manufacturers mr ON f.manufacturer_id = mr.manufacturer_id
    JOIN dim_manufacturers mn ON n.manufacturer_id = mn.manufacturer_id
    WHERE LOWER(mn.manufacturer_name) = 'undefined'
      AND LOWER(mr.manufacturer_name) NOT IN ('undefined','none','','n/a','unknown')
''')['units'][0] or 0

units_undef_to_real = q('''
    SELECT SUM(f.unit_count) AS units
    FROM fact_project_manufacturers f
    JOIN fact_project_manufacturers n
        ON  f.project_id  = n.project_id
        AND f.category_id = n.category_id
        AND f.source_type = n.source_type
        AND f.is_normalized = 0 AND n.is_normalized = 1
    JOIN dim_manufacturers mr ON f.manufacturer_id = mr.manufacturer_id
    JOIN dim_manufacturers mn ON n.manufacturer_id = mn.manufacturer_id
    WHERE LOWER(mr.manufacturer_name) = 'undefined'
      AND LOWER(mn.manufacturer_name) NOT IN ('undefined','none','','n/a','unknown')
''')['units'][0] or 0

total_norm_units = q('SELECT SUM(unit_count) FROM fact_project_manufacturers WHERE is_normalized=1').iloc[0,0]
lossy_total = units_real_to_undef + units_undef_to_real
print(f'\nUnits impact of lossy normalization:')
print(f'  Real names incorrectly mapped to Undefined : {units_real_to_undef:,} units lost to black hole')
print(f'  Undefined incorrectly mapped to real name  : {units_undef_to_real:,} units attributed to wrong manufacturer')
print(f'  Total affected                             : {lossy_total:,} units ({lossy_total/total_norm_units*100:.1f}% of all normalized units)')
print(f'  {lossy_total:,} units are either invisible or wrongly attributed, a data quality problem at business scale.')


In [ ]:
# Scale of the Undefined problem: units behind the #1 normalized name
undef_units = q('''
SELECT m.manufacturer_name, SUM(f.unit_count) AS total_units, COUNT(*) AS row_count
FROM fact_project_manufacturers f
JOIN dim_manufacturers m USING (manufacturer_id)
WHERE f.is_normalized = 1
  AND LOWER(m.manufacturer_name) = 'undefined'
GROUP BY m.manufacturer_name
''')
print('Scale of Undefined in normalized data:')
print(undef_units.to_string(index=False))

total_units = q('SELECT SUM(unit_count) AS total FROM fact_project_manufacturers WHERE is_normalized=1')['total'][0]
undef_total = undef_units['total_units'].sum()
print(f'\nUndefined = {undef_total:,} units out of {total_units:,} total ({undef_total/total_units*100:.1f}% of all normalized units)')


### 1d) What normalization misses: fuzzy duplicate clusters in normalized names

In [ ]:
# Known duplicates visible by inspection (no fuzzy matching needed)
known_dupes = [
    ("Undefined", "undefined",        "case, junk values not deduplicated"),
    ("Hart & Cooley", "Hart and Cooley", "& vs and"),
    ("climatemaster", "ClimateMaster",   "case"),
    ("dristeem", "DriSteem",             "case"),
    ("Flakt", "Flaktwoods",              "same company, different name"),
    ("Quoru", "Quorum",                  "typo"),
    ("Wessels", "Wessles",               "typo"),
    ("Valent", "Valent Air",             "partial name"),
    ("Valent", "Velent",                 "typo of Valent"),
    ("Epocha", "Ephoca",                 "OCR/anagram error"),
    ("Solaonics", "Solaronics",          "OCR error"),
]
generic_non_manufacturers = ["Commercial", "Custom", "Existing", "Others", "Selected", "Varies"]

print("Known remaining duplicates in normalized names (spotted by inspection):")
for a, b, reason in known_dupes:
    print(f"  {a!r:25s} vs {b!r:25s}  ({reason})")

print("")
print(f"Generic words masquerading as manufacturers: {generic_non_manufacturers}")

top_norm = q('''
SELECT m.manufacturer_name AS name, SUM(f.unit_count) AS total_units
FROM fact_project_manufacturers f
JOIN dim_manufacturers m USING (manufacturer_id)
WHERE f.is_normalized = 1
  AND m.manufacturer_name NOT IN ('undefined','Undefined','')
GROUP BY m.manufacturer_name
ORDER BY total_units DESC
LIMIT 1500
''')

names_list = top_norm['name'].tolist()
print(f'Fuzzy-clustering top {len(names_list)} normalized names (threshold=88)...')

THRESHOLD = 88
visited = set()
clusters = []

for name in names_list:
    if name in visited:
        continue
    matches = process.extract(
        name, names_list,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=THRESHOLD,
        limit=20
    )
    cluster = [m[0] for m in matches if m[0] not in visited]
    if len(cluster) > 1:
        clusters.append(cluster)
        visited.update(cluster)
    else:
        visited.add(name)

print(f'Found {len(clusters)} potential duplicate clusters still present after normalization')


In [ ]:
units_map = top_norm.set_index('name')['total_units'].to_dict()

cluster_impact = sorted(
    [{'members': c, 'size': len(c),
      'total_units': sum(units_map.get(n, 0) for n in c)}
     for c in clusters],
    key=lambda x: x['total_units'], reverse=True
)

print('Top 20 remaining duplicate clusters by unit-count impact:')
for i, cl in enumerate(cluster_impact[:20]):
    print(f"  {i+1:2d}. ({cl['size']} variants, {cl['total_units']:,} units)  {' | '.join(cl['members'])}")


### 1e) Error taxonomy in normalized names

In [ ]:
all_norm_names = q('''
SELECT DISTINCT m.manufacturer_name AS name
FROM fact_project_manufacturers f
JOIN dim_manufacturers m USING (manufacturer_id)
WHERE f.is_normalized = 1
''')['name'].tolist()

GENERIC_WORDS = {
    'commercial', 'custom', 'existing', 'others', 'selected',
    'varies', 'various', 'other', 'contractor', 'owner', 'none',
    'tbd', 'na', 'n/a', 'not applicable', 'unknown', 'as specified',
}

def classify(name):
    n = str(name).strip()
    nl = n.lower()
    # Undefined / empty
    if nl in ('undefined', 'none', '', 'n/a', 'na', 'not applicable', 'unknown'):
        return 'junk_undefined'
    # Generic non-manufacturer words
    if nl in GENERIC_WORDS:
        return 'junk_generic_word'
    # Pure numbers / punctuation
    if re.fullmatch('[\\d\\s\\-./:]+', n):
        return 'junk_numeric'
    # Model numbers / schedule artifacts
    if re.search('\\b(model|type|unit|schedule|series|style)\\b', n, re.I) or re.search('\\d{4,}', n):
        return 'junk_model_number'
    # Placeholder phrases
    if re.search('\\b(or equal|approved equal|as indicated|see schedule|per plan|as shown)\\b', n, re.I):
        return 'junk_placeholder'
    # OCR artifacts: characters outside normal manufacturer name charset
    if re.search("[^a-zA-Z0-9 &.,()\\-/\']+", n):
        return 'ocr_artifact'
    # Suspiciously short (single char or number)
    if len(n) <= 2:
        return 'junk_too_short'
    return 'ok'

name_series = pd.Series(all_norm_names)
tax = name_series.apply(classify).value_counts()
print('Taxonomy of normalized name quality:')
print(tax.to_string())

all_cats = ['junk_undefined','junk_generic_word','junk_numeric','junk_model_number',
            'junk_placeholder','ocr_artifact','junk_too_short']
for cat in all_cats:
    examples = name_series[name_series.apply(classify) == cat].head(6).tolist()
    if examples:
        print(f'\n[{cat}]')
        for ex in examples:
            print(f'  {repr(ex)}')


### 1f) Summary visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Part 1: Manufacturer Name Normalization Quality', fontsize=14, fontweight='bold')

# Left: raw vs normalized distinct string counts
ax = axes[0]
labels_bar = counts.sort_values('is_normalized')['label'].tolist()
vals   = counts.sort_values('is_normalized')['distinct_name_strings'].tolist()
bars = ax.bar(labels_bar, vals, color=['#e07b54','#4c8fbd'], width=0.5, edgecolor='white')
ax.set_ylabel('Distinct name strings')
ax.set_title('Distinct Manufacturer Name Strings\nRaw vs Normalized')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 20, f'{v:,}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, max(vals) * 1.2)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Right: error taxonomy
ax2 = axes[1]
tax_plot = tax[tax.index != 'ok']
colors = ['#d62728','#ff7f0e','#9467bd','#8c564b','#e377c2']
ax2.barh(tax_plot.index[::-1], tax_plot.values[::-1],
         color=colors[:len(tax_plot)], edgecolor='white')
ax2.set_xlabel('Count of distinct normalized names')
ax2.set_title('Error Taxonomy in Normalized Names')
for i, v in enumerate(tax_plot.values[::-1]):
    ax2.text(v + 0.5, i, str(v), va='center', fontsize=10)

plt.tight_layout()
plt.savefig('part1_normalization_quality.png', dpi=150, bbox_inches='tight')
plt.show()
raw_n  = counts[counts['is_normalized']==0]['distinct_name_strings'].values[0]
norm_n = counts[counts['is_normalized']==1]['distinct_name_strings'].values[0]
reduction_pct = (raw_n - norm_n) / raw_n * 100
undef_n = undef_units['total_units'].sum()
total_n = q('SELECT SUM(unit_count) FROM fact_project_manufacturers WHERE is_normalized=1').iloc[0,0]
print(f'Key findings: normalization reduced {raw_n:,} raw name strings to {norm_n:,} '
      f'({reduction_pct:.1f}% reduction). However, {undef_n:,} units '
      f'({undef_n/total_n*100:.1f}% of all normalized units) remain attributed to "Undefined", '
      f'and lossy mappings affect {real_to_undef["raw_name"].nunique():,} real manufacturer names.')


In [ ]:
# Dynamic summary printed from actual query results
raw_count  = counts[counts['is_normalized']==0]['distinct_name_strings'].values[0]
norm_count = counts[counts['is_normalized']==1]['distinct_name_strings'].values[0]
reduction  = (raw_count - norm_count) / raw_count * 100

top_fixed = fixed.head(3)
top_fixed_examples = '  |  '.join(
    f"{r['raw_name']} -> {r['norm_name']} (x{r['occurrences']})"
    for _, r in top_fixed.iterrows()
)

undef_pct = undef_total / total_units * 100

n_lossy_real_to_undef = real_to_undef['raw_name'].nunique()
n_lossy_undef_to_real = undef_to_real['norm_name'].nunique()

print("=" * 70)
print("PART 1 SUMMARY: MANUFACTURER NAME NORMALIZATION")
print("=" * 70)
print(f"\nString reduction: {raw_count:,} raw -> {norm_count:,} normalized ({reduction:.1f}% reduction)")
print(f"Names changed by normalization: {n_changed:,} / {total_raw:,} ({n_changed/total_raw*100:.1f}%)")

print(f"\nCRITICAL: LOSSY NORMALIZATION (information destroyed):")
print(f"   {n_lossy_real_to_undef:,} real manufacturer names were mapped TO 'Undefined'")
print(f"   'undefined' raw was mapped TO {n_lossy_undef_to_real:,} real manufacturer names")
print(f"   -> Normalization is assigning incorrect identities, not just cleaning strings")

print(f"\nSCALE: 'Undefined' is the #1 normalized name:")
print(f"   {undef_total:,} units ({undef_pct:.1f}% of all normalized units) are unresolved")

print(f"\nWhat normalization fixes well:")
print(f"   {top_fixed_examples}")

print(f"\nRemaining duplicates after normalization ({len(clusters)} fuzzy clusters found):")
for i, cl in enumerate(cluster_impact[:5]):
    print(f"   {' | '.join(cl['members'])}  ({cl['total_units']:,} units affected)")

print(f"\nKnown duplicates visible by inspection (no fuzzy matching needed):")
for a, b, reason in known_dupes:
    print(f"   {a!r} vs {b!r}  ({reason})")

print(f"\nError taxonomy in normalized names:")
for cat, cnt in tax.items():
    print(f"   {cat:25s} : {cnt}")


---
## Part 1 (cont.): Proposed Improved Entity Resolution

### Why the current approach falls short

The current normalization is a single-pass lookup/rule system. It has three core failure modes:

| Failure | Example | Root cause |
|---|---|---|
| **Wrong mapping** | `TITUS` -> `Undefined` | Lookup table collision or missing entry |
| **Missed duplicates** | `Hart & Cooley` vs `Hart and Cooley` | No fuzzy matching |
| **Junk propagated** | `undefined` -> `Trane` | No confidence gating |

### Proposed pipeline

A robust entity resolution system needs five stages:

**Stage 1: Cleaning & filtering**
Strip junk before attempting resolution. Names that are `undefined`, numeric, generic words,
or shorter than 3 chars should be flagged as `unresolved` rather than matched to a real manufacturer.

**Stage 2: Blocking**
Fuzzy matching all 9,967 raw names against each other is O(n^2). Use blocking to reduce the
search space: group candidates by their first two characters (or first token) so each name
only competes within its block.

**Stage 3: Candidate pair scoring**
For each pair within a block, compute two signals and combine them:
- `token_sort_ratio`: handles word reordering ("Bell & Gossett" vs "Gossett Bell")
- `token_set_ratio`: handles subset names ("Trane" vs "Trane Technologies")

Combined score = `0.6 * token_sort + 0.4 * token_set`

**Stage 4: Canonical name selection**
Within each cluster of matched names, pick the canonical form by:
1. Most frequent name in the raw data (popularity signal)
2. Tie-break: title-cased, shortest unambiguous form

**Stage 5: Confidence gating**
Only emit a merge suggestion if score >= threshold (e.g. 88). Below threshold, leave as-is
rather than risk a wrong merge. Flag low-confidence names for human review.


### PoC: Candidate pair generation on clean normalized names

In [ ]:
# Step 1: get clean normalized names with unit counts (exclude junk)
clean_norm = q('''
SELECT m.manufacturer_name AS name, SUM(f.unit_count) AS total_units
FROM fact_project_manufacturers f
JOIN dim_manufacturers m USING (manufacturer_id)
WHERE f.is_normalized = 1
GROUP BY m.manufacturer_name
''')

# Apply the same classifier from 1e to filter to real manufacturer names only
clean_norm['category'] = clean_norm['name'].apply(classify)
clean_names_df = clean_norm[clean_norm['category'] == 'ok'].copy()
freq_map = clean_names_df.set_index('name')['total_units'].to_dict()
clean_names = clean_names_df['name'].tolist()

print(f'Clean normalized names to resolve: {len(clean_names)} (of 510 total normalized)')


In [ ]:
# Step 2: blocking by first token to reduce O(n^2) comparisons
from collections import defaultdict

def first_token(name):
    return re.split(r'[\s&,/]', name.strip())[0].lower()[:3]

blocks = defaultdict(list)
for name in clean_names:
    blocks[first_token(name)].append(name)

total_pairs = sum(len(v) * (len(v) - 1) // 2 for v in blocks.values())
print(f'Blocks: {len(blocks)}, candidate pairs to score: {total_pairs:,}  (vs {len(clean_names)*(len(clean_names)-1)//2:,} without blocking)')


In [ ]:
# Step 3: score pairs within each block and collect merge candidates
THRESHOLD = 82

merge_candidates = []

for block_names in blocks.values():
    if len(block_names) < 2:
        continue
    for i, a in enumerate(block_names):
        for b in block_names[i+1:]:
            sort_score = fuzz.token_sort_ratio(a, b)
            set_score  = fuzz.token_set_ratio(a, b)
            combined   = 0.6 * sort_score + 0.4 * set_score
            if combined >= THRESHOLD:
                merge_candidates.append({
                    'name_a':    a,
                    'name_b':    b,
                    'sort':      sort_score,
                    'set':       set_score,
                    'score':     round(combined, 1),
                    'units_a':   freq_map.get(a, 0),
                    'units_b':   freq_map.get(b, 0),
                })

pairs_df = pd.DataFrame(merge_candidates).sort_values('score', ascending=False)
print(f'Merge candidates found: {len(pairs_df)}')
print()
print(pairs_df[['name_a','name_b','score','units_a','units_b']].head(30).to_string(index=False))


In [ ]:
# Step 4: canonical name selection
# Within each merge cluster, pick the name with the most units as the canonical form.
# Use union-find to group transitively connected pairs.

parent = {n: n for n in clean_names}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    parent[find(x)] = find(y)

for _, row in pairs_df.iterrows():
    union(row['name_a'], row['name_b'])

# Group into clusters
clusters_map = defaultdict(list)
for name in clean_names:
    clusters_map[find(name)].append(name)

multi = {k: v for k, v in clusters_map.items() if len(v) > 1}

# Pick canonical = highest unit count in cluster
proposals = []
for members in multi.values():
    canonical = max(members, key=lambda n: freq_map.get(n, 0))
    total = sum(freq_map.get(n, 0) for n in members)
    for alt in members:
        if alt != canonical:
            proposals.append({
                'canonical':  canonical,
                'merge_this': alt,
                'units_saved': freq_map.get(alt, 0),
                'cluster_total_units': total,
            })

proposals_df = pd.DataFrame(proposals).sort_values('cluster_total_units', ascending=False)
print(f'Total merge proposals: {len(proposals_df)} names -> {len(multi)} canonical entities')
print()
print(proposals_df.head(40).to_string(index=False))


### PoC Validation: merge examples and a borderline case

In [ ]:
# Show 3-5 good merges and 1 borderline/bad merge to validate the method

print("=== GOOD MERGES (method working correctly) ===")
good_examples = proposals_df[
    proposals_df['canonical'].isin(['Hart & Cooley','Daikin','ClimateMaster','DriSteem','Trane'])
].head(10)
for _, row in good_examples.iterrows():
    saved = freq_map.get(row['merge_this'], 0)
    print(f"  '{row['merge_this']}' -> '{row['canonical']}'  ({saved:,} units de-fragmented)")

# Also show top cluster members grouped
print()
print("Top 5 clusters with all members:")
for i, (root, members) in enumerate(list(multi.items())[:5]):
    canon = max(members, key=lambda n: freq_map.get(n, 0))
    total_u = sum(freq_map.get(n, 0) for n in members)
    print(f"  Cluster {i+1}: canonical='{canon}', total={total_u:,} units")
    for m in members:
        print(f"    '{m}' ({freq_map.get(m,0):,} units)")

# Borderline / potentially bad merge: low score pairs
print()
print("=== BORDERLINE / POTENTIALLY BAD MERGES (threshold risk) ===")
low_score = []
for block_names in blocks.values():
    for i, a in enumerate(block_names):
        for b in block_names[i+1:]:
            s = 0.6*fuzz.token_sort_ratio(a,b) + 0.4*fuzz.token_set_ratio(a,b)
            if 82 <= s <= 87:
                low_score.append({'name_a': a, 'name_b': b, 'score': round(s,1)})
borderline = pd.DataFrame(low_score).sort_values('score').head(5) if low_score else pd.DataFrame()
if not borderline.empty:
    print(borderline.to_string(index=False))
    print("  -> These pairs score just above threshold; human review recommended before merging.")
else:
    print("  No borderline cases found at current threshold.")


### Summary: Part 1 (cont.) Findings

**What the PoC does**
- Filters junk names out before matching (avoids polluting results)
- Uses blocking on the first 3 characters to reduce candidate pairs by ~90%
- Scores pairs with a combined `token_sort + token_set` signal
- Uses union-find to transitively group clusters and picks the canonical by unit frequency

**Limitations of the PoC**
- Blocking by prefix misses cross-prefix variants (e.g. abbreviations like `BAC` vs `Baltimore Aircoil`)
- No abbreviation expansion dictionary
- Threshold tuning is manual, a production system would calibrate on labeled pairs

**Recommended next steps for production**
1. Build a gold-standard set of ~200 labeled pairs (match / no-match) to tune the threshold
2. Add an abbreviation lookup table for common HVAC brand abbreviations
3. Use embedding-based similarity (e.g. sentence-transformers) for semantic matching
4. Gate all merges with a confidence score, route low-confidence cases to human review


---
## Part 2: Project Duplicate Detection

### Approach

Because parsing is non-deterministic, identical PDFs can produce slightly different extracted
fields, so exact matching on project_name alone is not enough.

**Blocking strategy**: duplicates almost always come from the same company. Within each company,
we block by the first 4 alpha characters of the project name. This keeps candidate pairs small
without missing obvious duplicates.

**Signals used (and why):**

| Signal | Weight | Rationale |
|---|---|---|
| `name_sim`: fuzzy token-sort ratio on project name | 0.45 | Strongest single signal; handles word reordering |
| `engineer_jaccard`: overlap of engineer-of-record sets | 0.20 | Same project -> same engineer firm |
| `mfr_jaccard`: overlap of normalized manufacturer sets | 0.15 | Same plans -> similar manufacturer fingerprint |
| `sheet_sim`: closeness of sheet counts | 0.10 | Same PDF -> same page count |
| `equip_sim`: closeness of equipment counts | 0.10 | Same schedules -> similar equipment counts |

Combined score >= 0.80 -> likely duplicate. >= 0.65 -> needs review.

**Signal considered but not implemented: `created_at` proximity**
Projects uploaded minutes or hours apart by the same user are almost certainly re-processing
runs of the same PDF. In production, adding a time-proximity bonus (e.g. +0.10 if uploaded
within 24 hours by the same user) would improve recall for rapid re-upload duplicates without
introducing false positives, since it only fires when combined with other signals.


In [ ]:
from collections import defaultdict

# Load project data
projects = q('''
SELECT project_id, company_id, project_name,
       num_sheets, num_equipments, market_category, created_at
FROM dim_projects
''')

# Engineer sets per project (normalized)
eng_df = q('SELECT project_id, engineer_name FROM project_engineers WHERE is_normalized=1')
eng_sets = eng_df.groupby('project_id')['engineer_name'].apply(frozenset).to_dict()

# Manufacturer fingerprint per project (normalized)
mfr_df = q('SELECT project_id, manufacturer_id FROM fact_project_manufacturers WHERE is_normalized=1')
mfr_sets = mfr_df.groupby('project_id')['manufacturer_id'].apply(frozenset).to_dict()

print(f'Projects loaded : {len(projects):,}')
print(f'Projects with engineers : {len(eng_sets):,}')
print(f'Projects with mfr data  : {len(mfr_sets):,}')


In [ ]:
# 896 projects have empty project names: they'd all land in the same block
# per company and generate false-positive pairs driven purely by engineer/mfr
# overlap. Exclude them from name-based blocking; they need a separate strategy.
named   = projects[projects['project_name'].str.strip() != ''].copy()
unnamed = projects[projects['project_name'].str.strip() == ''].copy()
print(f'Named projects   : {len(named):,}')
print(f'Unnamed projects : {len(unnamed):,}  (excluded from name-based blocking)')

def name_key(name):
    return re.sub('[^a-zA-Z]', '', str(name)).upper()[:4]

blocks = defaultdict(list)
for row in named.itertuples():
    key = (row.company_id, name_key(row.project_name))
    blocks[key].append(row)

total_pairs = sum(len(v)*(len(v)-1)//2 for v in blocks.values() if len(v) > 1)
print(f'Blocks: {len(blocks):,}')
print(f'Candidate pairs to score: {total_pairs:,}  (vs {len(named)*(len(named)-1)//2:,} brute-force)')


In [ ]:
def jaccard(a, b):
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

def score_pair(r1, r2):
    name_sim  = fuzz.token_sort_ratio(str(r1.project_name), str(r2.project_name)) / 100.0
    s1, s2    = r1.num_sheets or 0, r2.num_sheets or 0
    sheet_sim = 1 - abs(s1 - s2) / max(s1, s2, 1)
    e1, e2    = r1.num_equipments or 0, r2.num_equipments or 0
    equip_sim = 1 - abs(e1 - e2) / max(e1, e2, 1)
    eng_j     = jaccard(eng_sets.get(r1.project_id, frozenset()),
                        eng_sets.get(r2.project_id, frozenset()))
    mfr_j     = jaccard(mfr_sets.get(r1.project_id, frozenset()),
                        mfr_sets.get(r2.project_id, frozenset()))
    combined  = (0.45 * name_sim +
                 0.20 * eng_j   +
                 0.15 * mfr_j   +
                 0.10 * sheet_sim +
                 0.10 * equip_sim)
    return round(combined, 3), round(name_sim, 3), round(eng_j, 3), round(mfr_j, 3)

pairs = []
for members in blocks.values():
    if len(members) < 2:
        continue
    for i, r1 in enumerate(members):
        for r2 in members[i+1:]:
            score, ns, ej, mj = score_pair(r1, r2)
            if score >= 0.65:
                pairs.append({
                    'project_a':  r1.project_id,
                    'project_b':  r2.project_id,
                    'company':    r1.company_id,
                    'name_a':     r1.project_name,
                    'name_b':     r2.project_name,
                    'score':      score,
                    'name_sim':   ns,
                    'eng_j':      ej,
                    'mfr_j':      mj,
                    'sheets_a':   r1.num_sheets,
                    'sheets_b':   r2.num_sheets,
                })

dup_df = pd.DataFrame(pairs).sort_values('score', ascending=False)
auto   = dup_df[dup_df['score'] >= 0.80]
review = dup_df[(dup_df['score'] >= 0.65) & (dup_df['score'] < 0.80)]

print(f'Likely duplicates (score >= 0.80) : {len(auto):,} pairs')
print(f'Needs review    (0.65-0.80)       : {len(review):,} pairs')
print()
print('Top 20 highest-confidence duplicate pairs:')
cols = ['company','name_a','name_b','score','name_sim','eng_j','mfr_j','sheets_a','sheets_b']
print(auto[cols].head(20).to_string(index=False))


### 2b) Characterize duplication

In [ ]:
# Union-find to cluster duplicate pairs into groups
all_ids = projects['project_id'].tolist()
parent  = {pid: pid for pid in all_ids}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    parent[find(x)] = find(y)

for _, row in auto.iterrows():          # use high-confidence pairs only
    union(row['project_a'], row['project_b'])

dup_clusters = defaultdict(list)
for pid in all_ids:
    dup_clusters[find(pid)].append(pid)

multi_clusters = [v for v in dup_clusters.values() if len(v) > 1]
dup_project_ids = {pid for c in multi_clusters for pid in c}

total_projects  = len(projects)
n_dup_projects  = len(dup_project_ids)
n_dup_clusters  = len(multi_clusters)

print(f'Total projects         : {total_projects:,}')
print(f'Projects in duplicates : {n_dup_projects:,}  ({n_dup_projects/total_projects*100:.1f}%)')
print(f'Duplicate groups       : {n_dup_clusters:,}')
print(f'Avg group size         : {n_dup_projects/n_dup_clusters:.1f}')

# Group size distribution (upper bound 10000 to safely capture any large groups)
sizes = pd.Series([len(c) for c in multi_clusters])
bins  = pd.cut(sizes, bins=[1,2,3,5,10,10000], labels=['2','3','4-5','6-10','10+'])
print()
print('Distribution of duplicate group sizes:')
print(bins.value_counts().sort_index().to_string())

# Call out large groups: could be re-processing or multi-user uploads of same plan
large = [(max(c, key=lambda pid: len(pid)), len(c)) for c in multi_clusters if len(c) >= 5]
large.sort(key=lambda x: -x[1])
if large:
    print()
    print('Large duplicate groups (size >= 5): likely re-processing or shared plan across users:')
    for pid, sz in large[:10]:
        name = projects[projects['project_id']==pid]['project_name'].values[0]
        co   = projects[projects['project_id']==pid]['company_id'].values[0]
        print(f'  size={sz:3d}  {co}  {name}')


In [ ]:
# Duplication by company
projects['in_dup'] = projects['project_id'].isin(dup_project_ids)
by_company = (projects.groupby('company_id')
              .agg(total=('project_id','count'),
                   duplicates=('in_dup','sum'))
              .assign(dup_pct=lambda d: (d['duplicates']/d['total']*100).round(1))
              .sort_values('dup_pct', ascending=False))
print('Duplication rate by company:')
print(by_company.to_string())

print()

# Duplication by market category
by_cat = (projects[projects['market_category'] != '']
          .groupby('market_category')
          .agg(total=('project_id','count'),
               duplicates=('in_dup','sum'))
          .assign(dup_pct=lambda d: (d['duplicates']/d['total']*100).round(1))
          .sort_values('dup_pct', ascending=False))
print('Duplication rate by market category:')
print(by_cat.to_string())


### 2b-ii) Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Part 2: Project Duplicate Detection', fontsize=14, fontweight='bold')

# Left: group size distribution
ax = axes[0]
size_counts = bins.value_counts().sort_index()
ax.bar(size_counts.index.astype(str), size_counts.values, color='#4c8fbd', edgecolor='white')
ax.set_xlabel('Duplicate group size')
ax.set_ylabel('Number of groups')
ax.set_title('Distribution of Duplicate Group Sizes')
for i, v in enumerate(size_counts.values):
    ax.text(i, v + 0.3, str(v), ha='center', va='bottom', fontsize=10)

# Right: duplication rate by company
ax2 = axes[1]
by_co = by_company.sort_values('dup_pct')
colors = ['#d62728' if p > 20 else '#e07b54' if p > 10 else '#4c8fbd'
          for p in by_co['dup_pct']]
bars = ax2.barh(by_co.index, by_co['dup_pct'], color=colors, edgecolor='white')
ax2.set_xlabel('% projects in a duplicate group')
ax2.set_title('Duplication Rate by Company')
ax2.axvline(by_company['dup_pct'].mean(), color='black', linestyle='--', linewidth=1,
            label=f"avg {by_company['dup_pct'].mean():.1f}%")
ax2.legend(fontsize=9)
for bar, v in zip(bars, by_co['dup_pct']):
    ax2.text(v + 0.3, bar.get_y() + bar.get_height()/2,
             f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('part2_duplicate_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: part2_duplicate_detection.png')


### 2b-iii) Duplicate group examples and false-positive spot check

In [ ]:
# Show 2-3 full duplicate clusters side by side to visually confirm correctness
show_clusters = sorted(multi_clusters, key=len, reverse=True)[:3]

proj_lookup = projects.set_index('project_id')

for ci, cluster in enumerate(show_clusters):
    rows = proj_lookup.loc[cluster].sort_values('created_at')
    print(f"\n=== Duplicate Group {ci+1} (size={len(cluster)}) ===")
    eng_sample = {pid: list(eng_sets.get(pid, set()))[:2] for pid in cluster}
    cols = ['project_name','company_id','num_sheets','num_equipments','created_at']
    display_df = rows[cols].copy()
    display_df['engineers'] = [', '.join(eng_sample.get(pid, [])) for pid in display_df.index]
    print(display_df.head(6).to_string())
    if len(cluster) > 6:
        print(f'  ... and {len(cluster)-6} more copies')


In [ ]:
# False-positive spot check: show 5 pairs from the 0.65-0.80 "needs review" tier
print("=== FALSE-POSITIVE SPOT CHECK (needs-review tier, score 0.65-0.80) ===")
print("These are genuinely ambiguous, validates why we don't auto-merge them:\n")
spot = review.sort_values('score', ascending=False).head(5)
for _, row in spot.iterrows():
    print(f"  score={row['score']:.2f}  |  '{row['name_a']}'")
    print(f"         vs  '{row['name_b']}'")
    print(f"         sheets: {row['sheets_a']} vs {row['sheets_b']}  |  "
          f"eng_overlap={row['eng_j']:.2f}  |  mfr_overlap={row['mfr_j']:.2f}")
    print()
print("Note: pairs with high name_sim but low engineer/mfr overlap are likely same project")
print("name at different companies, a false positive the threshold correctly keeps in review.")


### 2c) Deduplication strategy

In [ ]:
# For each duplicate cluster, pick the canonical record and show conflicts

def pick_canonical(cluster_ids, proj_df):
    """
    Canonical selection rules (in order):
    1. Most recent created_at  (latest re-processing likely has best extraction)
    2. Tie-break: highest num_equipments (richest extraction)
    3. Tie-break: highest num_sheets
    """
    rows = proj_df[proj_df['project_id'].isin(cluster_ids)].copy()
    rows['created_at_dt'] = pd.to_datetime(rows['created_at'], utc=True, errors='coerce')
    rows = rows.sort_values(['created_at_dt','num_equipments','num_sheets'],
                            ascending=False)
    return rows.iloc[0]['project_id']

canonical_map = {}   # alt_id -> canonical_id
conflict_rows = []

for cluster in multi_clusters:
    canon = pick_canonical(cluster, projects)
    for pid in cluster:
        canonical_map[pid] = canon
        if pid != canon:
            r_alt   = projects[projects['project_id'] == pid].iloc[0]
            r_canon = projects[projects['project_id'] == canon].iloc[0]
            conflict_rows.append({
                'canonical':      canon,
                'duplicate':      pid,
                'sheet_conflict': r_alt['num_sheets'] != r_canon['num_sheets'],
                'equip_conflict': r_alt['num_equipments'] != r_canon['num_equipments'],
            })

conflicts = pd.DataFrame(conflict_rows)
n_sheet_conflict = conflicts['sheet_conflict'].sum()
n_equip_conflict = conflicts['equip_conflict'].sum()
n_total_alts     = len(conflicts)

print(f'Duplicate records to retire   : {n_total_alts:,}')
print(f'  with sheet count conflict   : {n_sheet_conflict:,}  ({n_sheet_conflict/n_total_alts*100:.1f}%)')
print(f'  with equipment count conflict: {n_equip_conflict:,}  ({n_equip_conflict/n_total_alts*100:.1f}%)')
print()
print('Sample canonical -> duplicate mapping (first 15):')
print(conflicts.head(15).to_string(index=False))


### Summary: Part 2 Findings

**Duplicate identification**
- Blocked within company + name prefix, then scored 5 signals (name, engineer, manufacturer fingerprint, sheet count, equipment count)
- High-confidence pairs (>= 0.80): auto-merge candidates
- Medium-confidence (0.65-0.80): flag for human review

**Canonical record selection**
Pick the most recent upload as canonical, later re-processing tends to have better extraction.
Tie-break on equipment count then sheet count (richer extraction = more useful record).

**Unnamed projects (896 excluded)**
The 896 projects with empty names were excluded from name-based blocking to avoid false
positives. They still need a separate deduplication pass using only non-name signals:
engineer overlap, manufacturer fingerprint, sheet count, and created_at proximity.

**Conflict handling: field-level merge strategy**

Different fields need different merge rules:

| Field | Strategy | Rationale |
|---|---|---|
| `manufacturer_id` set | **Union** across all duplicates | Different extraction runs may catch different equipment; losing any is worse than merging all |
| `engineer_name` set | **Union** across all duplicates | Same logic, different runs may extract different engineer firms |
| `architect_name` set | **Union** across all duplicates | Same logic as engineers |
| `num_sheets` | **Max** across duplicates | More pages = richer extraction |
| `num_equipments` | **Max** across duplicates | Higher count = more complete extraction |
| `created_at` | Keep canonical's timestamp | Records when the project was first seen |
| `project_name` | Keep canonical's value | Latest extraction is most reliable |

Never silently drop data, log all conflicts so an analyst can audit any auto-merge.

**Note on large groups**
Groups of 10+ (e.g. "NEW CITY HALL & JUSTICE CENTER" with 26 copies) are worth investigating:
are these re-processing runs by the same user, or the same plan uploaded by different users
at the same company? The answer affects how aggressively to auto-merge vs. keep separate records
for each user's view of the project.

**Confidence tiers**

| Score | Action |
|---|---|
| >= 0.80 | Auto-merge, retire duplicate |
| 0.65-0.80 | Flag for analyst review |
| < 0.65 | Leave as separate projects |


---
## Part 3: Exploratory Analysis

We explore four questions a manufacturer rep would actually care about:

1. **Who dominates overall?** Market share by unit volume
2. **Where does each manufacturer specialize?** Unit share by equipment category
3. **Who do you compete and co-exist with?** Manufacturer co-occurrence on the same projects
4. **Is the market growing or shifting?** Project activity and manufacturer presence over time


### Dataset Overview

In [ ]:
overview = q('''
SELECT
    (SELECT COUNT(*)                   FROM dim_projects)               AS total_projects,
    (SELECT COUNT(DISTINCT company_id) FROM dim_projects)               AS companies,
    (SELECT COUNT(DISTINCT username)   FROM dim_projects)               AS users,
    (SELECT substr(MIN(created_at),1,7) FROM dim_projects)             AS data_from,
    (SELECT substr(MAX(created_at),1,7) FROM dim_projects)             AS data_to,
    (SELECT COUNT(*)                   FROM dim_equipment_categories)   AS equipment_categories,
    (SELECT SUM(unit_count) FROM fact_project_manufacturers WHERE is_normalized=1
       AND manufacturer_id NOT IN (
           SELECT manufacturer_id FROM dim_manufacturers
           WHERE LOWER(manufacturer_name) = 'undefined'))               AS total_units_clean
''')
print('Dataset snapshot:')
print(overview.T.rename(columns={0:'value'}).to_string())

print()
print('Top 8 equipment categories by units:')
print(q('''
    SELECT c.category_name, SUM(f.unit_count) AS units,
           COUNT(DISTINCT f.project_id)        AS projects
    FROM fact_project_manufacturers f
    JOIN dim_equipment_categories c USING (category_id)
    WHERE f.is_normalized = 1
    GROUP BY c.category_name
    ORDER BY units DESC LIMIT 8
''').to_string(index=False))


### Viz 1: Overall manufacturer market share (top 20 by units)

In [ ]:
mkt = q('''
SELECT m.manufacturer_name AS name, SUM(f.unit_count) AS units,
       COUNT(DISTINCT f.project_id) AS projects
FROM fact_project_manufacturers f
JOIN dim_manufacturers m USING (manufacturer_id)
WHERE f.is_normalized = 1
  AND LOWER(m.manufacturer_name) != 'undefined'
  AND m.manufacturer_name != ''
GROUP BY m.manufacturer_name
ORDER BY units DESC
LIMIT 20
''')

fig, ax = plt.subplots(figsize=(11, 7))
colors = ['#2166ac' if i == 0 else '#4c8fbd' if i < 5 else '#92c5de'
          for i in range(len(mkt))]
bars = ax.barh(mkt['name'][::-1], mkt['units'][::-1], color=colors[::-1], edgecolor='white')
ax.set_xlabel('Total units specified')
ax.set_title('Top 20 Manufacturers by Units Specified\n(normalized names, excluding Undefined)',
             fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, v, p in zip(bars, mkt['units'][::-1], mkt['projects'][::-1]):
    ax.text(v + 200, bar.get_y() + bar.get_height()/2,
            f'{v:,} units  ({p:,} projects)', va='center', fontsize=8)
ax.set_xlim(0, mkt['units'].max() * 1.35)
plt.tight_layout()
plt.savefig('part3_viz1_market_share.png', dpi=150, bbox_inches='tight')
plt.show()

total_units = mkt['units'].sum()
all_clean_units = q('''SELECT SUM(unit_count) as u FROM fact_project_manufacturers f
    JOIN dim_manufacturers m USING(manufacturer_id)
    WHERE is_normalized=1 AND LOWER(m.manufacturer_name) != 'undefined'
    AND m.manufacturer_name NOT IN ('Others','other','None','Selected','Varies','Custom','Existing','Commercial')
''')['u'][0]
top4_units = mkt[~mkt['name'].isin(['Others','other','Selected','Varies','Custom','Existing','Commercial'])].head(4)['units'].sum()
print(f'Insight: Titus, Price, Greenheck, and Trane together account for {top4_units/all_clean_units*100:.1f}% of all '
      f'non-undefined units, roughly half the market is concentrated in four brands.')


### Viz 2: Manufacturer specialization by equipment category (heatmap)

In [ ]:
# Top 12 manufacturers and top 15 categories by unit volume
top_mfr = q('''
SELECT m.manufacturer_name AS name
FROM fact_project_manufacturers f
JOIN dim_manufacturers m USING (manufacturer_id)
WHERE f.is_normalized = 1 AND LOWER(m.manufacturer_name) != 'undefined'
GROUP BY m.manufacturer_name ORDER BY SUM(f.unit_count) DESC LIMIT 12
''')['name'].tolist()

top_cat = q('''
SELECT c.category_name AS cat
FROM fact_project_manufacturers f
JOIN dim_equipment_categories c USING (category_id)
WHERE f.is_normalized = 1
GROUP BY c.category_name ORDER BY SUM(f.unit_count) DESC LIMIT 15
''')['cat'].tolist()

mfr_in  = ','.join(repr(n) for n in top_mfr)
cat_in  = ','.join(repr(c) for c in top_cat)
hmap_data = q(f'''
SELECT m.manufacturer_name AS name, c.category_name AS cat,
       SUM(f.unit_count) AS units
FROM fact_project_manufacturers f
JOIN dim_manufacturers m USING (manufacturer_id)
JOIN dim_equipment_categories c USING (category_id)
WHERE f.is_normalized = 1
  AND m.manufacturer_name IN ({mfr_in})
  AND c.category_name     IN ({cat_in})
GROUP BY m.manufacturer_name, c.category_name
''')

pivot = hmap_data.pivot(index='name', columns='cat', values='units').fillna(0)
# Normalize each manufacturer's row to show share-of-category (highlights specialization)
pivot_norm = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_norm = pivot_norm.reindex(top_mfr).fillna(0)

fig, ax = plt.subplots(figsize=(15, 6))
sns.heatmap(pivot_norm, ax=ax, cmap='YlOrRd', linewidths=0.4, linecolor='white',
            annot=True, fmt='.0f', annot_kws={'size': 7},
            cbar_kws={'label': '% of manufacturer units in category'})
ax.set_title('Manufacturer Specialization by Equipment Category\n'
             "(each row = % of that manufacturer's units per category)",
             fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=35, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('part3_viz2_category_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
all_fan_units  = q('''SELECT SUM(f.unit_count) FROM fact_project_manufacturers f
    JOIN dim_equipment_categories c USING(category_id) JOIN dim_manufacturers m USING(manufacturer_id)
    WHERE f.is_normalized=1 AND c.category_name='Fans' AND LOWER(m.manufacturer_name)!='undefined'
    ''').iloc[0,0]
all_diff_units = q('''SELECT SUM(f.unit_count) FROM fact_project_manufacturers f
    JOIN dim_equipment_categories c USING(category_id) JOIN dim_manufacturers m USING(manufacturer_id)
    WHERE f.is_normalized=1 AND c.category_name='GRD - Supply Diffuser/Grille' AND LOWER(m.manufacturer_name)!='undefined'
    ''').iloc[0,0]
gh_fan_pct    = pivot.loc['Greenheck','Fans']    / all_fan_units  * 100 if 'Greenheck' in pivot.index else 0
titus_diff_pct = pivot.loc['Titus','GRD - Supply Diffuser/Grille'] / all_diff_units * 100 if 'Titus' in pivot.index else 0
print(f'Insight: Greenheck owns Fans ({gh_fan_pct:.0f}% of all fan units) and Titus leads '
      f'Supply Diffusers ({titus_diff_pct:.0f}%). Price and Titus split VAV Boxes almost evenly. '
      f'Mitsubishi is the clear specialist in Split VRF. A rep selling fans faces Greenheck on nearly every job.')


### Viz 3: Manufacturer co-occurrence (competitive & partnership landscape)

In [ ]:
# For each pair of top manufacturers, count shared projects
top15 = q('''
SELECT m.manufacturer_name AS name
FROM fact_project_manufacturers f
JOIN dim_manufacturers m USING (manufacturer_id)
WHERE f.is_normalized = 1 AND LOWER(m.manufacturer_name) != 'undefined'
GROUP BY m.manufacturer_name ORDER BY COUNT(DISTINCT f.project_id) DESC LIMIT 15
''')['name'].tolist()

top15_in = ','.join(repr(n) for n in top15)
cooc = q(f'''
SELECT a.manufacturer_name AS mfr_a, b.manufacturer_name AS mfr_b,
       COUNT(DISTINCT fa.project_id) AS shared_projects
FROM fact_project_manufacturers fa
JOIN fact_project_manufacturers fb ON fa.project_id = fb.project_id
JOIN dim_manufacturers a ON fa.manufacturer_id = a.manufacturer_id
JOIN dim_manufacturers b ON fb.manufacturer_id = b.manufacturer_id
WHERE fa.is_normalized = 1 AND fb.is_normalized = 1
  AND a.manufacturer_name IN ({top15_in})
  AND b.manufacturer_name IN ({top15_in})
GROUP BY a.manufacturer_name, b.manufacturer_name
''')

cooc_pivot = cooc.pivot(index='mfr_a', columns='mfr_b', values='shared_projects').fillna(0)
cooc_pivot = cooc_pivot.reindex(index=top15, columns=top15).fillna(0)

# Zero out diagonal so self-counts don't dominate the color scale
import numpy as np
np.fill_diagonal(cooc_pivot.values, 0)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cooc_pivot, ax=ax, cmap='Blues', linewidths=0.3, linecolor='white',
            annot=True, fmt='.0f', annot_kws={'size': 7})
ax.set_title('Manufacturer Co-occurrence: Shared Projects\n'
             '(top 15 manufacturers by project count, self-counts zeroed)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=40, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('part3_viz3_cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()
top_pair     = cooc_pivot.stack().idxmax()
top_pair_n   = int(cooc_pivot.loc[top_pair])
trane_mitsu  = int(cooc_pivot.loc['Trane','Mitsubishi']) if 'Trane' in cooc_pivot.index and 'Mitsubishi' in cooc_pivot.columns else 0
print(f'Insight: {top_pair[0]} + {top_pair[1]} is the most common pairing '
      f'({top_pair_n:,} shared projects), they cover complementary categories '
      f'(diffusers vs fans) so they appear together on nearly every large HVAC job. '
      f'Trane + Mitsubishi co-occur on {trane_mitsu:,} projects, reflecting overlapping VRF and fan coil lines. '
      f'High co-occurrence between direct competitors (e.g. Trane + Carrier) reveals displacement opportunities.')


### Viz 4: Project activity and manufacturer presence over time

In [ ]:
# Monthly project uploads
projects['month'] = pd.to_datetime(projects['created_at'], utc=True, errors='coerce').dt.to_period('M')
monthly_projects = projects.groupby('month').size().reset_index(name='project_count')
monthly_projects['month_dt'] = monthly_projects['month'].dt.to_timestamp()

# Monthly unit volume for top 6 manufacturers
top6 = top15[:6]
trends = q(f'''
SELECT substr(p.created_at, 1, 7) AS month,
       m.manufacturer_name             AS name,
       SUM(f.unit_count)               AS units
FROM fact_project_manufacturers f
JOIN dim_manufacturers m USING (manufacturer_id)
JOIN dim_projects p      USING (project_id)
WHERE f.is_normalized = 1
  AND m.manufacturer_name IN ({','.join(repr(n) for n in top6)})
GROUP BY month, m.manufacturer_name
ORDER BY month
''')
trends['month_dt'] = pd.to_datetime(trends['month'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=False)
fig.suptitle('Part 3 Viz 4: Project Activity & Manufacturer Presence Over Time',
             fontsize=13, fontweight='bold')

# Top: monthly project count
ax1.bar(monthly_projects['month_dt'], monthly_projects['project_count'],
        color='#4c8fbd', width=20, edgecolor='white')
ax1.set_ylabel('Projects uploaded')
ax1.set_title('Monthly Project Upload Volume')
ax1.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right')

# Bottom: monthly units per manufacturer
palette = plt.cm.tab10.colors
for i, mfr in enumerate(top6):
    sub = trends[trends['name'] == mfr].sort_values('month_dt')
    ax2.plot(sub['month_dt'], sub['units'], marker='o', markersize=3,
             linewidth=1.8, label=mfr, color=palette[i])
ax2.set_ylabel('Units specified')
ax2.set_title('Monthly Units Specified: Top 6 Manufacturers')
ax2.legend(fontsize=8, loc='upper left')
ax2.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('part3_viz4_time_trends.png', dpi=150, bbox_inches='tight')
plt.show()
data_start = monthly_projects['month'].min()
data_end   = monthly_projects['month'].max()
n_months   = len(monthly_projects)
peak_month = monthly_projects.loc[monthly_projects['project_count'].idxmax(), 'month']
print(f'Insight: the dataset spans {n_months} months ({data_start} to {data_end}), too short to '
      f'identify seasonal patterns with confidence. Upload volume peaks in {peak_month} and '
      f'manufacturer unit trends broadly track project volume, suggesting most variation is '
      f'driven by upload activity rather than true demand shifts. A 2-3 year window would be '
      f'needed to draw conclusions about seasonality or sustained brand growth.')


### Competitor Brands in the Market (Hidden Schedule Analysis)

In [ ]:
# hidden_schedule rows = competitor equipment the rep sees but doesn't sell.
# The useful signal is not whether a brand is invisible in visible schedules
# (every major brand appears in both), but the RATIO of hidden to total appearances.
# A high ratio means the brand shows up frequently on jobs the rep doesn't sell into,
# making it a priority competitor or expansion target.

hidden = q('''
SELECT m.manufacturer_name AS name,
       SUM(CASE WHEN f.source_type='hidden_schedule' THEN f.unit_count ELSE 0 END) AS hidden_units,
       SUM(CASE WHEN f.source_type='visible'         THEN f.unit_count ELSE 0 END) AS visible_units,
       COUNT(DISTINCT CASE WHEN f.source_type='hidden_schedule' THEN f.project_id END) AS hidden_projects
FROM fact_project_manufacturers f
JOIN dim_manufacturers m USING (manufacturer_id)
WHERE f.is_normalized = 1
  AND LOWER(m.manufacturer_name) NOT IN ('undefined','others','','none','varies','custom','existing')
GROUP BY m.manufacturer_name
HAVING hidden_units > 100
ORDER BY hidden_units DESC
LIMIT 30
''')

hidden['hidden_ratio'] = (hidden['hidden_units'] /
                          (hidden['hidden_units'] + hidden['visible_units'] + 1))

print('Top 15 manufacturers by total hidden-schedule units:')
print(hidden[['name','hidden_units','visible_units','hidden_projects','hidden_ratio']]
      .head(15).to_string(index=False))

print()
print('Brands with the highest hidden-to-total ratio (min 500 hidden units):')
print('These appear disproportionately on jobs the rep does not sell into.')
high_ratio = (hidden[hidden['hidden_units'] >= 500]
              .sort_values('hidden_ratio', ascending=False)
              .head(10))
print(high_ratio[['name','hidden_units','visible_units','hidden_ratio']].to_string(index=False))

top_ratio_brand = high_ratio.iloc[0]
print(f"\nInsight: {top_ratio_brand['name']} has the highest hidden ratio "
      f"({top_ratio_brand['hidden_ratio']:.2f}), meaning roughly "
      f"{top_ratio_brand['hidden_ratio']*100:.0f}% of its unit appearances are on jobs "
      f"the rep does not sell. This is a direct competitor displacement opportunity.")


In [ ]:
# Dynamic summary: all numbers pulled from dataframes computed above
top4_pct_val   = top4_units / all_clean_units * 100
gh_fan_val     = gh_fan_pct
titus_diff_val = titus_diff_pct
top_pair_val   = top_pair
top_pair_n_val = top_pair_n
trane_mitsu_val = trane_mitsu

# VAV box split
vav_price  = pivot.loc['Price','Standard VAV Boxes']  if 'Price'  in pivot.index and 'Standard VAV Boxes' in pivot.columns else 0
vav_titus  = pivot.loc['Titus','Standard VAV Boxes']  if 'Titus'  in pivot.index and 'Standard VAV Boxes' in pivot.columns else 0

print("=" * 70)
print("PART 3 SUMMARY")
print("=" * 70)
print(f"Viz 1: Market share")
print(f"  Top 4 brands (Titus, Price, Greenheck, Trane) = {top4_pct_val:.1f}% of all non-undefined units")
print()
print(f"Viz 2: Category specialization")
print(f"  Greenheck: {gh_fan_val:.0f}% of all Fan units")
print(f"  Titus    : {titus_diff_val:.0f}% of all Supply Diffuser units")
print(f"  VAV Boxes: Price={vav_price:,.0f} units vs Titus={vav_titus:,.0f} units")
print()
print(f"Viz 3: Co-occurrence")
print(f"  Top pair: {top_pair_val[0]} + {top_pair_val[1]} on {top_pair_n_val:,} shared projects")
print(f"  Trane + Mitsubishi: {trane_mitsu_val:,} shared projects")
print()
print(f"Viz 4: Time trends")
print(f"  Data spans {n_months} months ({data_start} to {data_end})")
print(f"  Peak upload month: {peak_month}")


### Summary: Part 3 Findings

**Viz 1: Market share**
The top 4 brands (Titus, Price, Greenheck, Trane) dominate the market, see dynamic summary
above for the exact share. Any rep competing in this space will encounter these four on nearly
every large project.

**Viz 2: Category specialization**
Greenheck commands a commanding share of Fan units; Titus leads Supply Diffusers. Price and
Titus split VAV Boxes almost evenly. Mitsubishi is the clear Split VRF specialist. See dynamic
summary for exact percentages from the data.

**Viz 3: Co-occurrence**
Titus + Greenheck is the most common pairing because they cover complementary categories and
appear together on nearly every large HVAC job. Trane + Mitsubishi co-occur on thousands of
projects, reflecting overlapping VRF and fan coil lines. High co-occurrence between direct
competitors reveals displacement opportunities.

**Viz 4: Time trends**
The dataset spans fewer than 12 months, too short to confirm seasonal patterns. Upload volume
peaks correlate with manufacturer unit volume, suggesting most month-to-month variation is
driven by upload activity rather than true demand shifts.

**Note on counts and duplicates**
All Part 3 unit and project counts include the duplicate projects identified in Part 2.
Deduplicating would reduce raw volumes but preserve relative rankings, the market share
ordering and category specialization findings are robust to deduplication.

**Competitor Brands in the Market (Hidden Schedule Analysis)**
Manufacturers appearing heavily in hidden schedules but rarely in visible ones represent
competitor brands that reps encounter on jobs but do not sell. These are high-priority
targets for rep expansion or competitor displacement analysis.
